In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datetime import datetime
import time
import utils
from utils import get_stock_data


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class base_RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size,
                 num_layers, dropout, lr, num_epochs, name,
                 rnn_type=nn.RNN,
                 dirpath='./logs/base_RNN'):
        
        super(base_RNN, self).__init__()
        
        self.name = name
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.lr = lr
        self.num_epochs = num_epochs
        self.mse_loss = nn.MSELoss()

        rnn_dropout = dropout if num_layers > 1 else 0
        self.model = rnn_type(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=rnn_dropout,
                            batch_first=True)

        self.out_layer = nn.Linear(hidden_size, output_size)
        self.optim = torch.optim.Adam(self.parameters(), lr=lr)
        self.dirpath = f"{dirpath}/{datetime.now().strftime('%Y-%m-%d')}"


    def forward(self, input):
        '''
        Passes input to model and grabs
        the hidden state at the final time step
        Returns the next closing price prediciton
        '''
        output, _ = self.model(input)
        final = self.out_layer(output[:, -1, :])
        return final


    def mape(self, preds, targets):
        '''
        Calculates the Mean Absolute Pecentage Error
        '''
        return float(torch.mean(torch.abs((preds - targets) / targets)).item() * 100)


    def fit(self, train_loader, stock_name=''):
        '''
        train model for an RNN-type model
        '''
        self.train(mode=True) #for dropout
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        train_results = utils.get_results_dict('train')

        start = time.time()
        #training loop with batches
        for epoch in range(self.num_epochs):
            epoch_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = self(X_batch)
                loss = self.mse_loss(output, y_batch)

                self.optim.zero_grad()
                loss.backward()
                self.optim.step()

                epoch_loss += loss.item()

            fit_time = time.time() - start
            avg_loss = epoch_loss / len(train_loader)

            train_results['avg_mse_loss'].append(avg_loss)
            train_results['stock_name'].append(stock_name)
            train_results['timestamp'].append(timestamp)
            train_results['epoch'].append(epoch)
            train_results['fit_time'].append(fit_time)

        #track training time + log results
        total_fit_time = time.time() - start
        utils.log(train_results, f'{self.dirpath}/{self.name}/all_{self.name}_train_results.csv')

        return total_fit_time, train_results


    def evaluate(self, test_loader, scaler, stock_name='', split='test'):
        '''
        test model for an RNN-type model
        '''
        self.eval() #for dropout
        all_preds = []
        all_targets = []

        start = time.time()
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                preds = self(X_batch)
                all_preds.append(preds.cpu()) #to cpu to apply to inverse_transform() later
                                              
                all_targets.append(y_batch)
        eval_time = time.time() - start

        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)

        #inverse transform the data from normalized back to prices
        all_preds = torch.tensor(scaler.inverse_transform(all_preds.numpy()))
        all_targets = torch.tensor(scaler.inverse_transform(all_targets.numpy()))

        mse = self.mse_loss(all_preds, all_targets).item()
        rmse = mse ** 0.5
        mape = self.mape(all_preds, all_targets)

        #log results
        eval_results = utils.get_results_dict('eval')
        eval_results['stock_name'] = [stock_name]
        eval_results['split'] = [split]
        eval_results['timestamp'] = [datetime.now().strftime('%Y-%m-%d %H:%M:%S')]
        eval_results['rmse'] = [rmse]
        eval_results['mape'] = [mape]
        eval_results['eval_time'] = [eval_time]
        eval_results['input_size'] = [self.input_size]
        eval_results['hidden_size'] = [self.hidden_size]
        eval_results['output_size'] = [self.output_size]
        eval_results['num_layers'] = [self.num_layers]
        eval_results['dropout'] = [self.dropout]
        eval_results['lr'] = [self.lr]
        eval_results['num_epochs'] = [self.num_epochs]
        utils.log(eval_results, f'{self.dirpath}/{self.name}/eval_summary/eval_results.csv')

        self.train()
        return rmse, mape, all_preds, all_targets

    @classmethod
    def train_test_eval(cls, cfg, model_type):
        '''
        train, test, and results for an RNN-type model
        '''
        print(f"\n{'-'*40} Starting... {model_type} {'-'*40}")
        start_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        ds, stocks = get_stock_data()

        train_size = int(len(ds) * 0.80)
        test_dates = ds.index[train_size:]

        loss_results = []
        pred_results = []

        #create an individual model on each stock
        for stock in stocks:
            X_train, X_test, y_train, y_test, scaler = utils.preprocess(ds, stock)
            #print(f"\n{'-'*40} Training: {stock} {'-'*40}")

            train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=False)
            test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=32, shuffle=False)

            model = cls(
                input_size=cfg['input_size'],
                hidden_size=cfg['hidden_size'],
                output_size=cfg['output_size'],
                num_layers=cfg['num_layers'],
                dropout=cfg['dropout'],
                lr=cfg['lr'],
                num_epochs=cfg['num_epochs'],
                name=cfg['name'],
            ).to(device)

            #train the model
            train_time, train_log = model.fit(train_loader, stock_name=stock)

            loss_results.append((stock, train_log['epoch'], train_log['avg_mse_loss']))

            #save model
            os.makedirs(f'./models/{cls.__name__}/{stock}', exist_ok=True)
            torch.save(model.state_dict(), f'./models/{cls.__name__}/{stock}/{stock}_{cfg["name"]}.pt')

            #test the models predictions
            train_rmse, train_mape, all_preds, all_targets = model.evaluate(train_loader, scaler=scaler, stock_name=stock, split='train')
            test_rmse, test_mape, preds, targets = model.evaluate(test_loader, scaler=scaler, stock_name=stock, split='test')
            pred_results.append((stock, test_dates, targets.numpy().flatten(), preds.numpy().flatten()))

            summary = {
                'stock_name': [stock],
                'train_rmse': [round(train_rmse, 2)],
                'test_rmse': [round(test_rmse,  2)],
                'train_mape': [round(train_mape, 2)],
                'test_mape': [round(test_mape,  2)],
                'fit_time': [round(train_time, 1)],
                'name': [cfg['name']],
                'hidden_size': [cfg['hidden_size']],
                'num_layers': [cfg['num_layers']],
                'dropout': [cfg['dropout']],
                'lr': [cfg['lr']],
                'num_epochs': [cfg['num_epochs']],
            }
            utils.log(summary, f'{model.dirpath}/{cfg["name"]}/{stock}/summary_results.csv')
            print(f'{stock} | Train RMSE: ${train_rmse:.2f} | Test RMSE: ${test_rmse:.2f} | Train MAPE: {train_mape:.2f}% | Test MAPE: {test_mape:.2f}% | Train time: {train_time:.1f}s')

        #plot all results in on a grid and save
        utils.plot_grid_training_loss(loss_results,
            save_path=f'{model.dirpath}/{cfg["name"]}/{model_type}_{cfg["name"]}_allstocks_train_loss.png',
            cfg=cfg)
        utils.plot_grid_true_v_pred(pred_results,
            save_path=f'{model.dirpath}/{cfg["name"]}/{model_type}_{cfg["name"]}_allstocks_true_vs_pred.png',
            cfg=cfg)

        end_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f'Began: {start_timestamp} | Finished: {end_timestamp}')


In [ ]:
class GRU(base_RNN):
    def __init__(self, input_size, hidden_size, output_size,
                 num_layers, dropout, lr, num_epochs, name,
                 dirpath='./logs/GRU'):
        
        super(GRU, self).__init__(input_size, hidden_size,
                                       output_size,num_layers,
                                       dropout, lr, num_epochs,
                                       name, dirpath=dirpath,
                                       rnn_type=nn.GRU)

In [ ]:
class LSTM(base_RNN):
    def __init__(self, input_size, hidden_size, output_size,
                 num_layers, dropout, lr, num_epochs, name,
                 dirpath='./logs/LSTM'):
        
        super(LSTM, self).__init__(input_size, hidden_size,
                                       output_size,num_layers,
                                       dropout, lr, num_epochs,
                                       name, dirpath=dirpath,
                                       rnn_type=nn.LSTM)

In [ ]:
NUM_EPOCHS = 50

In [ ]:
cfg1 = {
        'input_size' : 1,
        'hidden_size' : 64,
        'output_size' : 1,
        'num_layers' : 1,
        'dropout' : 0,
        'lr' : 0.001,
        'num_epochs' : NUM_EPOCHS,
        'name' : 'std_cfg'
      }

base_RNN.train_test_eval(cfg1, model_type='RNN')
GRU.train_test_eval(cfg1, model_type='GRU')
LSTM.train_test_eval(cfg1, model_type='LSTM')

In [ ]:
cfg2 = {
        'input_size' : 1,
        'hidden_size' : 64,
        'output_size' : 1,
        'num_layers' : 2,
        'dropout' : 0,
        'lr' : 0.001,
        'num_epochs' : NUM_EPOCHS,
        'name' : 'std_stacked'
      }

base_RNN.train_test_eval(cfg2, model_type='RNN')
GRU.train_test_eval(cfg2, model_type='GRU')
LSTM.train_test_eval(cfg2, model_type='LSTM')

In [ ]:
cfg2_dp = {
        'input_size' : 1,
        'hidden_size' : 64,
        'output_size' : 1,
        'num_layers' : 2,
        'dropout' : 0.2,
        'lr' : 0.001,
        'num_epochs' : NUM_EPOCHS,
        'name' : 'std_stacked_dp_20'
      }

base_RNN.train_test_eval(cfg2_dp, model_type='RNN')
GRU.train_test_eval(cfg2_dp, model_type='GRU')
LSTM.train_test_eval(cfg2_dp, model_type='LSTM')

In [ ]:
cfg128 = {
        'input_size' : 1,
        'hidden_size' : 128,
        'output_size' : 1,
        'num_layers' : 2,
        'dropout' : 0.2,
        'lr' : 0.001,
        'num_epochs' : NUM_EPOCHS,
        'name' : 'std_stacked_128'
      }

base_RNN.train_test_eval(cfg128, model_type='RNN')
GRU.train_test_eval(cfg128, model_type='GRU')
LSTM.train_test_eval(cfg128, model_type='LSTM')

In [ ]:
cfg32 = {
        'input_size' : 1,
        'hidden_size' : 32,
        'output_size' : 1,
        'num_layers' : 2,
        'dropout' : 0.2,
        'lr' : 0.001,
        'num_epochs' : NUM_EPOCHS,
        'name' : 'lr_01_stacked'
      }

base_RNN.train_test_eval(cfg32, model_type='RNN')
GRU.train_test_eval(cfg32, model_type='GRU')
LSTM.train_test_eval(cfg32, model_type='LSTM')

In [ ]:
cfglr_01 = {
        'input_size' : 1,
        'hidden_size' : 64,
        'output_size' : 1,
        'num_layers' : 2,
        'dropout' : 0.2,
        'lr' : 0.01,
        'num_epochs' : NUM_EPOCHS,
        'name' : 'std_stacked_32'
      }

base_RNN.train_test_eval(cfglr_01, model_type='RNN')
GRU.train_test_eval(cfglr_01, model_type='GRU')
LSTM.train_test_eval(cfglr_01, model_type='LSTM')